# （參考解答）06_financial_mathematics_and_option_pricing

> 這是對應主 notebook 的**完整參考解答版**。建議先自己完成主notebook 的練習，再對照本檔。所有解說與解答皆為本專案原創。

# Week 6 — 財務數學與選擇權定價

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 計算折現因子、現值與債券價格。
- 繪製 call、put 與簡單組合的 payoff 圖。
- 用 binomial tree 為歐式選擇權定價。
- 對 strike、波動度、到期、利率做敏感度分析。

## 預估學習時間

約 8–10 小時。

## 先備概念

- 基本代數與指數
- Week 1 的報酬概念

## 外部學習資源

- [NTU OpenCourseWare 基礎財金素養](https://ocw.aca.ntu.edu.tw/courses/110S204)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### 貨幣的時間價值

未來的現金流要先**折現**才能和今天的錢比較。年利率 $r$、$t$ 年後、每年複利 $m$ 次的折現因子為 $(1 + r/m)^{-mt}$。現值是各期現金流乘上折現因子後的總和。

### 選擇權 payoff 與 binomial 定價

歐式 call/put 到期 payoff 為 $\max(S-K,0)$、$\max(K-S,0)$。

本路線圖**刻意只用 binomial tree**：它只需要算術與 no-arbitrage 概念，**不需要** stochastic calculus，也**不要求** Black-Scholes 推導。選擇權價值是其到期 payoff 在 risk-neutral 機率下的折現期望值。

> 提醒：模型價格**不應**被解讀為對真實市場價格的預測。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.finance.fixed_income import (
    bond_price, discount_factor, present_value, zero_coupon_bond_price,
)
from quant_math_roadmap.finance.derivatives import (
    binomial_european_option, call_payoff, put_payoff,
    long_straddle_payoff, put_call_parity_gap,
)

### 現值計算器

In [ ]:
rate = 0.04
cash_flows = [100, 100, 100, 1100]  # 4 年期、年付息
times = [1, 2, 3, 4]
pv = present_value(cash_flows, times, rate)
print(f'折現率 {rate:.0%} 下，現金流現值 = {pv:.2f}')
for t in times:
    print(f'  t={t}: 折現因子 = {discount_factor(rate, t):.4f}')

### 債券定價：價格隨殖利率下降

In [ ]:
yields = np.linspace(0.01, 0.10, 50)
prices = [bond_price(1000, 0.05, 10, y, coupons_per_year=2) for y in yields]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yields, prices, label='10 年期、5% 票息債券')
ax.axhline(1000, linestyle='--', label='面額 = 1000')
ax.set_title('債券價格 vs 殖利率')
ax.set_xlabel('殖利率 (yield to maturity)')
ax.set_ylabel('債券價格')
ax.legend()
plt.show()
zcb = zero_coupon_bond_price(1000, 10, 0.05)
print(f'10 年期零息債券（殖利率 5%）價格 = {zcb:.2f}')

殖利率上升，債券價格下降；票息率等於殖利率時，債券以面額（par）定價。

### Duration 與 Convexity：債券的利率敏感度

知道價格如何隨殖利率變動，比知道單一價格更有用：

- **Macaulay duration**：現金流以現值加權的平均到達時間（年）。零息債券的 duration 恰等於到期年限。
- **Modified duration** $D_{mod} = -\frac{1}{P}\frac{dP}{dy}$：殖利率每變動 1 個百分點，價格大約變動 $D_{mod}$%。
- **Convexity** $C = \frac{1}{P}\frac{d^2P}{dy^2}$：曲率修正，二階近似為 $\Delta P/P \approx -D_{mod}\Delta y + \tfrac12 C (\Delta y)^2$。

In [ ]:
from quant_math_roadmap.finance.fixed_income import (
    bond_convexity, macaulay_duration, modified_duration,
)

args = dict(face_value=1000, coupon_rate=0.05,
            years_to_maturity=10, yield_to_maturity=0.04)
mac = macaulay_duration(**args)
mod = modified_duration(**args)
conv = bond_convexity(**args)
print(f'Macaulay duration = {mac:.4f} 年')
print(f'Modified duration = {mod:.4f}')
print(f'Convexity         = {conv:.4f}')

# 用「真實重新定價 vs 一階/二階近似」驗證這兩個數字的意義
p0 = bond_price(1000, 0.05, 10, 0.04, coupons_per_year=2)
dy = 0.01  # 殖利率 +100bp
p1 = bond_price(1000, 0.05, 10, 0.04 + dy, coupons_per_year=2)
actual = p1 / p0 - 1
first_order = -mod * dy
second_order = -mod * dy + 0.5 * conv * dy**2
print(f'實際價格變動    = {actual:+.4%}')
print(f'一階(duration)  = {first_order:+.4%}')
print(f'二階(+convexity)= {second_order:+.4%}  <- 更貼近實際')

零息債券檢查：`macaulay_duration(1000, 0.0, 5, 0.04, coupons_per_year=1)` 會精確回傳 5.0 年——唯一一筆現金流就在到期日。

### Payoff 圖

In [ ]:
spot_grid = np.linspace(50, 150, 200)
strike = 100.0
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(spot_grid, call_payoff(spot_grid, strike))
axes[0].set_title('Call payoff (K=100)')
axes[1].plot(spot_grid, put_payoff(spot_grid, strike))
axes[1].set_title('Put payoff (K=100)')
axes[2].plot(spot_grid, long_straddle_payoff(spot_grid, strike))
axes[2].set_title('Long straddle payoff (K=100)')
for ax in axes:
    ax.set_xlabel('到期標的價格 S')
    ax.set_ylabel('payoff')
plt.tight_layout()
plt.show()

### Binomial 歐式選擇權定價

In [ ]:
params = {'spot': 100.0, 'strike': 100.0, 'rate': 0.05,
          'volatility': 0.20, 'maturity': 1.0}
call = binomial_european_option(**params, n_steps=300, option_type='call')
put = binomial_european_option(**params, n_steps=300, option_type='put')
print(f'歐式 call 價格 = {call:.4f}')
print(f'歐式 put  價格 = {put:.4f}')
gap = put_call_parity_gap(call, put, params['spot'], params['strike'],
                          params['rate'], params['maturity'])
print(f'put-call parity 殘差 = {gap:.6f}  (應接近 0)')

### 敏感度分析

In [ ]:
vols = np.linspace(0.05, 0.6, 40)
call_by_vol = [binomial_european_option(100, 100, 0.05, v, 1.0,
               n_steps=200) for v in vols]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(vols, call_by_vol, label='ATM call (S=K=100)')
ax.set_title('歐式 call 價格 vs 波動度')
ax.set_xlabel('波動度代理值 sigma')
ax.set_ylabel('call 價格')
ax.legend()
plt.show()
print('波動度越高，選擇權越貴 — 因為更大的不確定性對買方有利。')

### 美式選擇權：一個 max 的差別

美式選擇權可以**提前履約**。在 binomial tree 上，這只是把每個節點的回溯值改成 `max(繼續持有的折現期望, 立刻履約的內含價值)`。兩個經典結論可以直接數值驗證：

1. **無股利標的的美式 call = 歐式 call**（提前履約永遠不划算）；
2. **美式 put ≥ 歐式 put**（深價內時提早收到履約金有時間價值）。

In [ ]:
from quant_math_roadmap.finance.derivatives import binomial_american_option

common = dict(spot=100.0, strike=110.0, rate=0.06,
              volatility=0.2, maturity=2.0, n_steps=300)
eu_call = binomial_european_option(option_type='call', **common)
am_call = binomial_american_option(option_type='call', **common)
eu_put = binomial_european_option(option_type='put', **common)
am_put = binomial_american_option(option_type='put', **common)
print(f'歐式 call = {eu_call:.4f} | 美式 call = {am_call:.4f}  (相等)')
print(f'歐式 put  = {eu_put:.4f} | 美式 put  = {am_put:.4f}  (美式較貴)')
print(f'美式 put 的提前履約溢價 = {am_put - eu_put:.4f}')

### Greeks：價格對每個輸入的敏感度

**Greeks** 回答「輸入動一點，價格動多少」。`binomial_greeks()` 用有限差分直接在樹上估計：

| Greek | 定義 | 直覺 |
|-------|------|------|
| delta | ∂V/∂S | 標的漲 1 元，選擇權漲多少 |
| gamma | ∂²V/∂S² | delta 本身變多快 |
| vega  | ∂V/∂σ | 波動度升 1 單位的影響 |
| theta | −∂V/∂T | 時間流逝一年的損耗 |
| rho   | ∂V/∂r | 利率升 1 單位的影響 |

> 樹上的有限差分是「近似的近似」，數字會有小幅抖動——這是教學工具，不是生產級定價器。

In [ ]:
from quant_math_roadmap.finance.derivatives import binomial_greeks

greeks_call = binomial_greeks(100, 100, 0.05, 0.2, 1.0, option_type='call')
greeks_put = binomial_greeks(100, 100, 0.05, 0.2, 1.0, option_type='put')
for name in ['delta', 'gamma', 'vega', 'theta', 'rho']:
    print(f'{name:>6}: call = {greeks_call[name]:>9.4f} | put = {greeks_put[name]:>9.4f}')
print()
print(f'delta_call - delta_put = {greeks_call['delta'] - greeks_put['delta']:.6f}'
      '  (put-call parity 預言這個差恰為 1)')

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用一句話解釋為什麼未來的錢要折現。
2. 為什麼債券價格與殖利率反向變動？
3. 解釋 long straddle 的 payoff 形狀，以及它在押注什麼。

### 應用練習

In [ ]:
# 應用練習 1：計算 binomial call 價格隨 strike 變化的曲線（其他參數固定），
# 並確認 strike 越高、call 越便宜。
strikes = np.linspace(80, 120, 20)
call_by_strike = [binomial_european_option(100, k, 0.05, 0.2, 1.0,
                  n_steps=150) for k in strikes]
print('strike 越高 call 越便宜?', all(np.diff(call_by_strike) < 0))

In [ ]:
# 應用練習 2：驗證 binomial 步數增加時 call 價格收斂（穩定下來）。
for steps in [10, 50, 200, 800]:
    price = binomial_european_option(100, 100, 0.05, 0.2, 1.0,
                                     n_steps=steps)
    print(f'n_steps={steps:>3}: call = {price:.4f}')
print('步數越多，價格越穩定（收斂）。')

### 反思問題

1. binomial 模型價格與真實市場價格幾乎不會完全相同。這對「用模型價格設計交易策略」有什麼提醒？

## 小測驗（自我檢核）
回答下面的選擇題，然後執行下一格自動對答案。答案以雜湊儲存，不會直接洩漏。

**Q1. 殖利率上升時，債券價格？**
- A. 上升
- B. 下降
- C. 不變
- D. 視票息而定

**Q2. 零息債券的 Macaulay duration 等於？**
- A. 0
- B. 到期年限
- C. 殖利率
- D. 票面利率

**Q3. 無股利標的的美式 call 相對歐式 call 的價格？**
- A. 較高
- B. 相等
- C. 較低
- D. 不一定

**Q4. convexity 為正代表什麼？**
- A. 殖利率下跌時的漲幅大於 duration 線性估計
- B. 價格與殖利率成正比
- C. 債券有違約風險
- D. duration 為負

In [ ]:
my_answers = {1: 'B', 2: 'B', 3: 'B', 4: 'A'}

import hashlib as _hashlib
_expected = {1: 'e572273e26af1c17', 2: '9ec4573f1c035f2f', 3: '32d10537ba48c5ce', 4: '28fd59f43f42474e'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: 未作答')
        continue
    _h = _hashlib.sha256(f'qmr-w6-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ 正確' if _ok else '✘ 不正確'))
print(f'得分: {_n_correct} / {len(my_answers)}')

### 解析

- **Q1 → B**：未來現金流以更高利率折現，現值必然下降——債券定價第一定律。
- **Q2 → B**：只有一筆到期現金流，加權平均時間就是到期年限本身。
- **Q3 → B**：提前履約放棄時間價值且無股利可收，永遠不划算，所以兩者等價。
- **Q4 → A**：價格-殖利率曲線向下凸：跌得比線性少、漲得比線性多。

## 常見錯誤

- **折現時搞錯複利頻率（年複利 vs 半年複利）。**
- **把選擇權 payoff（到期才實現）與選擇權現價混為一談。**
- **binomial 步數太少就把價格當精確值。**
- **宣稱模型價格等於真實市場價格。**

## 完成本週後，你應該能做到什麼

- [ ] 能計算現值與債券價格。
- [ ] 能繪製並解讀 call/put/straddle 的 payoff 圖。
- [ ] 能用 binomial tree 為歐式選擇權定價並解釋每一步。
- [ ] 能對主要參數做敏感度分析。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../../docs/math/) 與 [`docs/finance/`](../../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。